# NAME: SOWMYA HARDAGERI
# SRN : PES2UG23CS590

# Unit 2 Assignment: Building a Mixture of Experts (MoE) Router

# "Smart Customer Support Router"

Install Dependencies

In [1]:
!pip install groq python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.3 MB/s eta 0:00:00


Setup Environment + Groq Client

In [2]:
import os
import getpass
from groq import Groq

# Set API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Initialize client
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq client initialized successfully ✅")

Enter your Groq API Key: ··········
Groq client initialized successfully ✅


Define Experts (MODEL_CONFIG)

In [7]:
MODEL_NAME = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Technical Support Expert.
Be precise, structured, and code-focused.
If debugging, explain the issue clearly and provide corrected code snippets.
Avoid unnecessary empathy. Focus on solutions.
"""
    },

    "billing": {
        "system_prompt": """
You are a Billing Support Specialist.
Be empathetic, polite, and policy-driven.
Explain billing issues clearly and guide the customer through next steps.
Avoid technical jargon.
"""
    },

    "general": {
        "system_prompt": """
You are a helpful Customer Support Assistant.
Answer politely and clearly.
If unsure, provide general guidance.
"""
    }
}

print("Experts configured:", list(MODEL_CONFIG.keys()))

Experts configured: ['technical', 'billing', 'general']


Router Function (Core Task)

In [8]:
def route_prompt(user_input):
    routing_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general]

Return ONLY the category name.

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category

Orchestrator (Main MoE Logic)

In [9]:
def process_request(user_input):

    # Step 1: Route
    category = route_prompt(user_input)

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    # Step 2: Expert Response
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    final_answer = response.choices[0].message.content.strip()

    return {
        "category": category,
        "response": final_answer
    }

Test Cases

In [10]:
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hi there! What services do you offer?"
]

for query in test_queries:
    result = process_request(query)
    print("\n==============================")
    print("User Query:", query)
    print("Routed To:", result["category"])
    print("Response:\n", result["response"])


User Query: My python script is throwing an IndexError on line 5.
Routed To: technical
Response:
 **Error Analysis**

To troubleshoot the issue, I'll need more information about your script and the error message. Please provide the following:

1. The complete error message, including any stack traces.
2. The relevant code snippet surrounding line 5.
3. The expected input or data structure being used.

With this information, I can help you identify and resolve the issue.

**Example Input**

If you're providing a code snippet, please use Markdown formatting to make it easier to read. For example:

```python
# Relevant code snippet
my_list = [1, 2, 3]
try:
    # Line 5
    print(my_list[5])
except IndexError as e:
    print(f"Error: {e}")
```

**Possible Causes**

Based on the `IndexError` exception, here are some common causes:

* Accessing an index that is out of range for a list or other sequence.
* Attempting to access a key that does not exist in a dictionary.
* Other similar indexi

# BONUS — Tool Use Expert (Bitcoin Price)

Add Tool Expert

In [11]:
def fetch_bitcoin_price():
    # Mock API call
    return "$64,250 (Mock Data)"

def route_prompt_with_tool(user_input):

    routing_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general, crypto_tool]

Return ONLY the category name.

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    return response.choices[0].message.content.strip().lower()

Orchestrator with Tool

In [12]:
def process_request_with_tool(user_input):

    category = route_prompt_with_tool(user_input)

    if category == "crypto_tool":
        price = fetch_bitcoin_price()
        return {
            "category": category,
            "response": f"The current Bitcoin price is {price}."
        }

    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return {
        "category": category,
        "response": response.choices[0].message.content.strip()
    }

Test Tool

In [13]:
crypto_query = "What is the current price of Bitcoin?"

result = process_request_with_tool(crypto_query)

print("User Query:", crypto_query)
print("Routed To:", result["category"])
print("Response:", result["response"])

User Query: What is the current price of Bitcoin?
Routed To: crypto_tool
Response: The current Bitcoin price is $64,250 (Mock Data).


Conclusion:
For real-time data queries such as cryptocurrency prices, the system routes to a dedicated tool function instead of generating a hallucinated LLM response. This demonstrates tool-augmented routing and prevents misinformation, making the architecture more robust and production-ready.